# 05 · Báo cáo 2 (Slides A–H): Phân tích Chuyên sâu (`MBD_MINI_DEEP_INSIGHTS_DECK.pdf`)

Notebook tái lập và kiểm chứng trực tiếp **100% số liệu của 8 slide (A–H)** trong **Báo cáo 2 (`MBD_MINI_DEEP_INSIGHTS_DECK.pdf`)**, tuân thủ quy ước:
- **Cutoff:** `00:00` ngày mùng 1 tháng nhãn `T` theo giờ Moscow (`Europe/Moscow`).
- **Tập quan sát chuẩn (`Common Cohort`):** `100.224` khách × `10` mốc dự báo (`02/2022–11/2022`) = **`1.002.240` quan sát khách–tháng** có đủ nhãn `H=1` (`11.420` ca, `1,14%`) và `H=3` (`30.749` ca, `3,07%`).
- **Nguồn dữ liệu nền:** `all_customer_months.parquet` (`analyze_full_population_temporal_eda.py`), `event_type_monthly_profiles.parquet` & `geohash_4_monthly_profiles.parquet` (`extend_eda_monthly_features.py`), `monthly_segment_cards.csv` (`audit_insights_checklist.py`), và bảng `targets`.

In [ ]:
from pathlib import Path
import subprocess
import sys
import duckdb
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import display, Image, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
BASE_DIR = ROOT / "outputs/full_population_temporal_eda_20261118"
FIG_DIR = ROOT / "outputs/eda_figures_moscow"
INSIGHT_DIR = ROOT / "outputs/eda_insights_20261119"
SLIDE_DIR = INSIGHT_DIR / "slides_clean"

# Tải bảng panel tháng Moscow + hồ sơ event_type, geohash_4 và 4 nhãn targets
panel = pd.read_parquet(BASE_DIR / "all_customer_months.parquet")
panel["month"] = pd.to_datetime(panel["month"])

et_prof = pd.read_parquet(FIG_DIR / "event_type_monthly_profiles.parquet")
et_prof["month"] = pd.to_datetime(et_prof["month"])
et_prof = et_prof.rename(columns={c: f"et_{c}" for c in et_prof.columns if c not in ("client_id", "month")})

g4_prof = pd.read_parquet(FIG_DIR / "geohash_4_monthly_profiles.parquet")
g4_prof["month"] = pd.to_datetime(g4_prof["month"])
g4_prof = g4_prof.rename(columns={c: f"g4_{c}" for c in g4_prof.columns if c not in ("client_id", "month")})

con = duckdb.connect()
targets_glob = (ROOT / "data/MBD-mini/targets/**/*.parquet").as_posix()
targets = con.execute(f"""
    SELECT client_id, date_trunc('month', CAST(mon AS DATE))::TIMESTAMP AS month,
           target_1, target_2, target_3, target_4
    FROM read_parquet('{targets_glob}')
""").df()
targets["month"] = pd.to_datetime(targets["month"])

df = (
    panel.merge(et_prof[["client_id", "month", "et_unique_cells", "et_new_cells"]], on=["client_id", "month"], how="left")
         .merge(g4_prof[["client_id", "month", "g4_unique_cells", "g4_new_cells"]], on=["client_id", "month"], how="left")
         .merge(targets, on=["client_id", "month"], how="left")
         .sort_values(["client_id", "month"])
         .reset_index(drop=True)
)

g = df.groupby("client_id", sort=False)

# Đặc trưng tháng liền trước (T-1) và tích lũy nghiêm ngặt trước cutoff (< T)
df["trx_prev"] = g["trx_events"].shift(1).fillna(0)
df["geo_prev"] = g["geo_events"].shift(1).fillna(0)
df["dlg_prev"] = g["dialog_events"].shift(1).fillna(0)
df["et_unique_prev"] = g["et_unique_cells"].shift(1).fillna(0)
df["et_new_prev"] = g["et_new_cells"].shift(1).fillna(0)
df["g4_unique_prev"] = g["g4_unique_cells"].shift(1).fillna(0)
df["g4_new_prev"] = g["g4_new_cells"].shift(1).fillna(0)

df["trx_hist"] = g["trx_events"].cumsum() - df["trx_events"].fillna(0)
df["trx_active_months_hist"] = g["trx_events"].transform(lambda s: (s.fillna(0) > 0).cumsum()) - (df["trx_events"].fillna(0) > 0).astype(int)
df["trx_cum_before_prev"] = df["trx_hist"] - df["trx_prev"]
df["trx_months_before_prev"] = df["trx_active_months_hist"] - (df["trx_prev"] > 0).astype(int)
df["trx_prior_mean"] = np.where(df["trx_months_before_prev"] > 0, df["trx_cum_before_prev"] / df["trx_months_before_prev"], np.nan)
df["trx_surge_ratio"] = np.where(df["trx_prior_mean"] > 0, df["trx_prev"] / df["trx_prior_mean"], np.nan)

# Số tháng lịch sử (history_age_months): từ tháng đầu tiên xuất hiện bất kỳ sự kiện nào (< T) đến tháng T
any_ev = (df["trx_events"].fillna(0) > 0) | (df["geo_events"].fillna(0) > 0) | (df["dialog_events"].fillna(0) > 0)
first_any = df["month"].where(any_ev).groupby(df["client_id"], sort=False).transform("min")
# Đảm bảo chỉ tính các sự kiện nghiêm ngặt trước tháng T
df["first_any_before"] = first_any.where(first_any < df["month"])
df["history_age_months"] = (
    (df["month"].dt.year - df["first_any_before"].dt.year) * 12
    + (df["month"].dt.month - df["first_any_before"].dt.month)
)

# Lịch sử mua quan sát được trước tháng T
df["y_fill"] = df["y"].fillna(0)
df["prior_buys"] = g["y_fill"].cumsum() - df["y_fill"]

# Nhãn H=1, H=3 trên Common Cohort (10 mốc 02/2022 - 11/2022)
df["y2_lead"] = g["y"].shift(-1)
df["y3_lead"] = g["y"].shift(-2)
cohort = df[df["y"].notna() & df["y2_lead"].notna() & df["y3_lead"].notna()].copy()
cohort["h1"] = cohort["y"].astype(int)
cohort["h3"] = ((cohort["y"] > 0) | (cohort["y2_lead"] > 0) | (cohort["y3_lead"] > 0)).astype(int)

BASE_H1 = cohort["h1"].mean()
BASE_H3 = cohort["h3"].mean()
print(f"Common cohort: {len(cohort):,} rows | H1={cohort['h1'].sum():,} ({BASE_H1:.4%}) | H3={cohort['h3'].sum():,} ({BASE_H3:.4%})")
assert len(cohort) == 1_002_240 and cohort["h1"].sum() == 11_420 and cohort["h3"].sum() == 30_749

## Slide A · Giải mã đường cong chữ U & Bóc tách nhóm `0 Giao dịch` (`84.670` dòng)

In [ ]:
# 1. Đường cong chữ U theo tổng TRX tích lũy trước cutoff
cohort["trx_band"] = pd.cut(
    cohort["trx_hist"],
    bins=[-1, 0, 9, 99, np.inf],
    labels=["0 giao dịch", "1 - 9", "10 - 99", "≥ 100"],
)
u_tbl = cohort.groupby("trx_band", observed=True).agg(
    n=("h1", "size"), buyers_h1=("h1", "sum"), rate_h1=("h1", "mean"), rate_h3=("h3", "mean")
)
u_tbl["share_%"] = u_tbl["n"] / len(cohort) * 100
u_tbl["lift_h1"] = u_tbl["rate_h1"] / BASE_H1
display(u_tbl)

# 2. Bóc tách bên trong 84.670 dòng có trx_hist == 0
z0 = cohort[cohort["trx_hist"] == 0].copy()
z0["sub_group"] = np.select(
    [z0["dlg_prev"] > 0, (z0["dlg_prev"] == 0) & (z0["trx_events"] > 0)],
    ["1. Có Hội thoại tháng trước (T-1)", "2. Kích hoạt TRX mới (chưa DIALOG)"],
    default="3. Ngủ đông thực sự",
)
z0_tbl = z0.groupby("sub_group").agg(
    n=("h1", "size"), buyers_h1=("h1", "sum"), rate_h1=("h1", "mean"), rate_h3=("h3", "mean")
)
z0_tbl["share_in_0trx_%"] = z0_tbl["n"] / len(z0) * 100
z0_geo4 = z0[z0["g4_unique_prev"] >= 4]
print(f"Chỉ báo phụ trước cutoff (GEO >= 4 vùng trong 0 TRX): n={len(z0_geo4):,} ({len(z0_geo4)/len(z0):.2%}), H1={z0_geo4['h1'].mean():.2%}, H3={z0_geo4['h3'].mean():.2%}")
display(z0_tbl)

## Slide B · Tách bạch Tần suất (`TRX/tháng`) × Số tháng lịch sử & Tỷ số `TRX(T-1) / TB trước đó`

In [ ]:
# 1. Cố định nhóm 0 lần mua trước (loại tháng đầu 02/2022 chưa biết lịch sử nhãn) & 0 DIALOG ở T-1 và có lịch sử TRX > 0
b_sub = cohort[(cohort["month"] > "2022-02-01") & (cohort["prior_buys"] == 0) & (cohort["dlg_prev"] == 0) & (cohort["trx_hist"] > 0) & (cohort["history_age_months"] >= 1)].copy()
b_sub["trx_per_month"] = b_sub["trx_hist"] / b_sub["history_age_months"]
b_sub["vel_band"] = pd.cut(b_sub["trx_per_month"], bins=[0, 2, 5, 10, 25, np.inf], labels=["≤2", "2-5", "5-10", "10-25", ">25"])
b_sub["age_band"] = pd.cut(b_sub["history_age_months"], bins=[0, 6, 12, np.inf], labels=["1-6 tháng", "7-12 tháng", "≥13 tháng"])
vel_pivot = b_sub.pivot_table(index="vel_band", columns="age_band", values="h1", aggfunc="mean", observed=True) * 100
display(vel_pivot.round(2))

# 2. Tỷ số TRX(T-1) / Trung bình tháng trước đó
cohort["surge_group"] = np.select(
    [
        cohort["trx_hist"] == 0,
        (cohort["trx_cum_before_prev"] > 0) & (cohort["trx_prev"] == 0),
        (cohort["trx_cum_before_prev"] == 0) & (cohort["trx_prev"] > 0),
        cohort["trx_surge_ratio"] < 0.5,
        cohort["trx_surge_ratio"] <= 1.5,
        cohort["trx_surge_ratio"] > 1.5,
    ],
    ["0 TRX toàn lịch sử", "0× (Tạm dừng T-1)", "Trước T-1 = 0, T-1 > 0", "< 0,5×", "0,5× - 1,5×", "> 1,5×"],
    default="other",
)
surge_tbl = cohort.groupby("surge_group").agg(n=("h1", "size"), buyers_h1=("h1", "sum"), rate_h1=("h1", "mean"))
surge_tbl["share_%"] = surge_tbl["n"] / len(cohort) * 100
surge_tbl["lift_h1"] = surge_tbl["rate_h1"] / BASE_H1
display(surge_tbl)

## Slide C · Kiểm định đa tầng `DIALOG` theo tầng `TRX` & Lịch sử mua (Bác bỏ Nghịch lý Simpson)

In [ ]:
c_sub = cohort[(cohort["month"] > "2022-02-01") & (cohort["history_age_months"] >= 13)].copy()
c_sub["has_prior_buy"] = np.where(c_sub["prior_buys"] > 0, "≥1 lần mua trước", "0 lần mua trước")
c_sub["has_dlg_prev"] = c_sub["dlg_prev"] > 0
c_tbl = (
    c_sub.groupby(["has_prior_buy", "trx_band", "has_dlg_prev"], observed=True)
         .agg(n=("h1", "size"), buyers_h1=("h1", "sum"), rate_h1=("h1", "mean"), rate_h3=("h3", "mean"))
         .unstack("has_dlg_prev")
)
c_tbl["lift_dialog_vs_no"] = c_tbl[("rate_h1", True)] / c_tbl[("rate_h1", False)]
display(c_tbl.loc[[("0 lần mua trước", "1 - 9"), ("0 lần mua trước", "10 - 99"), ("0 lần mua trước", "≥ 100"), ("≥1 lần mua trước", "≥ 100")]])

## Slide D · Phân rã Shift-Share (`04/2022 → 11/2022`) & Độ phủ `DIALOG` vs Lift theo tháng

In [ ]:
# 1. Phân rã Shift-Share trên 6 phân khúc rời nhau (TRX state × DIALOG T-1)
cohort["trx_state"] = np.select(
    [cohort["trx_hist"] == 0, cohort["trx_prev"] > 0],
    ["no_trx_history", "trx_active_prev_month"],
    default="trx_older_history",
)
cohort["dlg_state"] = cohort["dlg_prev"] > 0

m0 = cohort[cohort["month"] == "2022-04-01"].groupby(["trx_state", "dlg_state"]).agg(n0=("h1", "size"), r0=("h1", "mean"))
m1 = cohort[cohort["month"] == "2022-11-01"].groupby(["trx_state", "dlg_state"]).agg(n1=("h1", "size"), r1=("h1", "mean"))
ss = m0.join(m1)
ss["w0"] = ss["n0"] / ss["n0"].sum()
ss["w1"] = ss["n1"] / ss["n1"].sum()
ss["mix_effect_pp"] = (ss["w1"] - ss["w0"]) * ss["r0"] * 100
ss["within_effect_pp"] = ss["w0"] * (ss["r1"] - ss["r0"]) * 100
ss["interaction_pp"] = (ss["w1"] - ss["w0"]) * (ss["r1"] - ss["r0"]) * 100
display(ss[["w0", "r0", "w1", "r1", "mix_effect_pp", "within_effect_pp", "interaction_pp"]].round(4))
print(
    f"Tổng Shift-Share (điểm %): Mix={ss['mix_effect_pp'].sum():+.3f}%, "
    f"Within={ss['within_effect_pp'].sum():+.3f}%, "
    f"Interaction={ss['interaction_pp'].sum():+.3f}%, "
    f"Total={(ss['mix_effect_pp'] + ss['within_effect_pp'] + ss['interaction_pp']).sum():+.3f}%"
)

# 2. Độ phủ DIALOG & Lift theo từng tháng (10 mốc)
monthly_seg = pd.read_csv(INSIGHT_DIR / "monthly_segment_cards.csv")
dlg_m = monthly_seg[(monthly_seg["feature"] == "dialog_recent") & (monthly_seg["group"] == "True") & (monthly_seg["horizon"] == 1)]
display(dlg_m[["month", "population_share", "rate", "lift"]])

## Slide E · Phân tích Đa nhãn: Ma trận Chuyển tiếp (`A → B`) & Đồng mua cùng tháng (`647` ca)

In [ ]:
# 1. Ma trận chuyển tiếp 4x4 trên 11 tháng có lịch sử nhãn (03/2022 - 01/2023, 1.102.464 dòng)
tdf = targets.sort_values(["client_id", "month"]).copy()
tg = tdf.groupby("client_id", sort=False)
for k in range(1, 5):
    tdf[f"prior_t{k}"] = (tg[f"target_{k}"].cumsum() - tdf[f"target_{k}"]) > 0

t_11 = tdf[tdf["month"] >= "2022-03-01"].copy()
assert len(t_11) == 1_102_464

rate_mat = pd.DataFrame(index=[f"Target {i}" for i in range(1, 5)], columns=[f"Target {j}" for j in range(1, 5)], dtype=float)
lift_mat = rate_mat.copy()
for i in range(1, 5):
    sub = t_11[t_11[f"prior_t{i}"]]
    for j in range(1, 5):
        p = sub[f"target_{j}"].mean()
        rate_mat.loc[f"Target {i}", f"Target {j}"] = p * 100
        lift_mat.loc[f"Target {i}", f"Target {j}"] = p / t_11[f"target_{j}"].mean()
display(Markdown("**Tỷ lệ mua Target B khi đã từng mua Target A (%)**"))
display(rate_mat.round(2))
display(Markdown("**Hệ số Lift (A → B)**"))
display(lift_mat.round(2))

# 2. Đồng mua gộp cùng tháng trên 12 tháng nhãn
tdf["n_prod"] = tdf[["target_1", "target_2", "target_3", "target_4"]].sum(axis=1)
buyers_all = (tdf["n_prod"] >= 1).sum()
co_all = (tdf["n_prod"] >= 2).sum()
print(f"Tổng tháng mua: {buyers_all:,} | Đồng mua >=2 SP cùng tháng: {co_all:,} ({co_all / buyers_all:.2%})")
pairs = []
for i in range(1, 5):
    for j in range(i + 1, 5):
        pairs.append({"pair": f"Target {i} + Target {j}", "count": int(((tdf[f"target_{i}"] == 1) & (tdf[f"target_{j}"] == 1)).sum())})
display(pd.DataFrame(pairs).sort_values("count", ascending=False))

## Slide F & G · Phát hiện mới về `TRX` (`event_type` mới/đa dạng) & Mối liên hệ độc lập `TRX × GEO`

In [ ]:
# 1. Số mã event_type mới ở T-1 (Slide F trái)
cohort["et_novelty"] = np.select(
    [cohort["trx_prev"] == 0, cohort["et_new_prev"] == 0, cohort["et_new_prev"] == 1],
    ["0 TRX ở T-1", "0 mã mới", "1 mã mới"],
    default="≥ 2 mã mới",
)
f1 = cohort.groupby("et_novelty").agg(n=("h1", "size"), buyers_h1=("h1", "sum"), rate_h1=("h1", "mean"), rate_h3=("h3", "mean"))
f1["share_%"] = f1["n"] / len(cohort) * 100
f1["lift_h1"] = f1["rate_h1"] / BASE_H1
display(f1.loc[["0 TRX ở T-1", "0 mã mới", "1 mã mới", "≥ 2 mã mới"]])

# 2. Số loại event_type khác nhau ở T-1 (Slide F phải)
cohort["et_div"] = np.select(
    [cohort["trx_prev"] == 0, cohort["et_unique_prev"] == 1, cohort["et_unique_prev"].between(2, 3), cohort["et_unique_prev"].between(4, 6)],
    ["0 TRX ở T-1", "1 loại", "2 - 3 loại", "4 - 6 loại"],
    default="≥ 7 loại",
)
f2 = cohort.groupby("et_div").agg(n=("h1", "size"), buyers_h1=("h1", "sum"), rate_h1=("h1", "mean"))
f2["share_%"] = f2["n"] / len(cohort) * 100
f2["lift_h1"] = f2["rate_h1"] / BASE_H1
display(f2.loc[["1 loại", "2 - 3 loại", "4 - 6 loại", "≥ 7 loại"]])

# 3. Số vùng geohash_4 ở T-1 (Slide G trái) + Lift riêng cho SP2 (Tín dụng)
cohort["g4_band"] = np.select(
    [cohort["geo_prev"] == 0, cohort["g4_unique_prev"] == 1, cohort["g4_unique_prev"].between(2, 4), cohort["g4_unique_prev"].between(5, 9)],
    ["0 vùng", "1 vùng", "2 - 4 vùng", "5 - 9 vùng"],
    default="≥ 10 vùng",
)
g1 = cohort.groupby("g4_band").agg(n=("h1", "size"), rate_h1=("h1", "mean"), rate_p2=("target_2", "mean"))
g1["share_%"] = g1["n"] / len(cohort) * 100
g1["lift_h1"] = g1["rate_h1"] / BASE_H1
g1["lift_p2"] = g1["rate_p2"] / cohort["target_2"].mean()
display(g1.loc[["0 vùng", "1 vùng", "2 - 4 vùng", "5 - 9 vùng", "≥ 10 vùng"]])

# 4. Độc lập TRX vs GEO (Spearman) & Ma trận tương tác 4x3 (Slide G phải)
active_both = cohort[(cohort["trx_prev"] > 0) & (cohort["geo_prev"] > 0)]
r_new = spearmanr(active_both["et_new_prev"], active_both["g4_unique_prev"]).statistic
print(f"Spearman(et_new_prev, g4_unique_prev) trên {len(active_both):,} dòng hoạt động cả 2 nguồn: r = {r_new:.4f}")

cohort["geo_3tier"] = np.select(
    [cohort["geo_prev"] == 0, cohort["g4_unique_prev"].between(1, 3)],
    ["0 vùng GEO", "1 - 3 vùng GEO"],
    default="≥ 4 vùng GEO",
)
inter = cohort.pivot_table(index="et_novelty", columns="geo_3tier", values="h1", aggfunc="mean") * 100
display(inter.loc[["0 TRX ở T-1", "0 mã mới", "1 mã mới", "≥ 2 mã mới"], ["0 vùng GEO", "1 - 3 vùng GEO", "≥ 4 vùng GEO"]].round(2))

## Slide H · Xuất lại bộ 8 Slide & PDF `MBD_MINI_DEEP_INSIGHTS_DECK.pdf`

In [ ]:
REBUILD_PDF = False
if REBUILD_PDF:
    subprocess.run([sys.executable, str(ROOT / "src/scripts/build_clean_presentation.py")], cwd=ROOT, check=True)

for idx in range(1, 9):
    img = SLIDE_DIR / f"slide_{idx:02d}.png"
    if img.exists():
        display(Markdown(f"### Slide {idx:02d}"))
        display(Image(filename=str(img), width=900))